In [ ]:
%cd /kaggle/working
!cp -r /kaggle/input/datasets/anfs2003/ragtruth-lettucedetect/lettucedetect_code/LettuceDetect .
%cd LettuceDetect
!pip install -q -e .

In [2]:
from kaggle_secrets import UserSecretsClient
import os
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")

In [3]:
%cd /kaggle/working/LettuceDetect
!python scripts/evaluate.py \
    --model_path anfs4554/lettucedetect-full-large \
    --data_path /kaggle/input/datasets/anfs2003/ragtruth-lettucedetect/ragtruth_data.json \
    --evaluation_type example_level \
    --dump_dir /kaggle/working/ragtruth_full_preds
!python /kaggle/input/datasets/anfs2003/ragtruth-lettucedetect/bootstrap_ragtruth.py \
    --dump_dir /kaggle/working/ragtruth_full_preds \
    --model full --out /kaggle/working/ragtruth_full_cis.csv

/kaggle/working/LettuceDetect

Evaluating model on test samples: 2700
config.json: 2.09kB [00:00, 4.99MB/s]
model.safetensors: 100%|████████████████████| 1.58G/1.58G [00:10<00:00, 147MB/s]
Loading weights: 100%|█| 174/174 [00:00<00:00, 1318.31it/s, Materializing param=
tokenizer_config.json: 100%|███████████████████| 351/351 [00:00<00:00, 1.33MB/s]
tokenizer.json: 3.58MB [00:00, 39.3MB/s]

Task type: Summary

Evaluating model on 900 samples

---- Example-Level Evaluation ----
Evaluating (Example Level):   0%|                       | 0/113 [00:00<?, ?it/s]W0716 00:53:54.104000 165 torch/_inductor/utils.py:1679] [1/0_1] Not enough SMs to use max_autotune_gemm mode
                                                                                
Detailed Example-Level Classification Report:
              precision    recall  f1-score   support

   Supported     0.8526    0.9555    0.9011       696
Hallucinated     0.7417    0.4363    0.5494       204

    accuracy                         0

In [4]:
%cd /kaggle/working/LettuceDetect
!python scripts/evaluate.py \
    --model_path anfs4554/lettucedetect-qa-only-large \
    --data_path /kaggle/input/datasets/anfs2003/ragtruth-lettucedetect/ragtruth_data.json \
    --evaluation_type example_level \
    --dump_dir /kaggle/working/ragtruth_qa_preds
!python /kaggle/input/datasets/anfs2003/ragtruth-lettucedetect/bootstrap_ragtruth.py \
    --dump_dir /kaggle/working/ragtruth_qa_preds \
    --model qa --out /kaggle/working/ragtruth_qa_cis.csv

/kaggle/working/LettuceDetect

Evaluating model on test samples: 2700
config.json: 2.09kB [00:00, 3.82MB/s]
model.safetensors: 100%|████████████████████| 1.58G/1.58G [00:12<00:00, 127MB/s]
Loading weights: 100%|█| 174/174 [00:00<00:00, 1336.01it/s, Materializing param=
tokenizer_config.json: 100%|███████████████████| 351/351 [00:00<00:00, 1.15MB/s]
tokenizer.json: 3.58MB [00:00, 29.1MB/s]

Task type: Summary

Evaluating model on 900 samples

---- Example-Level Evaluation ----
                                                                                
Detailed Example-Level Classification Report:
              precision    recall  f1-score   support

   Supported     0.8432    0.9037    0.8724       696
Hallucinated     0.5649    0.4265    0.4860       204

    accuracy                         0.7956       900
   macro avg     0.7040    0.6651    0.6792       900
weighted avg     0.7801    0.7956    0.7848       900


Evaluation Results:

Hallucination Detection (Class 1):
  Precis

In [5]:
import csv
exp = {('full','QA'):0.7138, ('full','Summary'):0.5494, ('full','Data2txt'):0.8833, ('full','whole'):0.7912,
       ('qa','QA'):0.6797, ('qa','Summary'):0.4860, ('qa','Data2txt'):0.7716, ('qa','whole'):0.7071}
for f, m in [('ragtruth_full_cis.csv','full'), ('ragtruth_qa_cis.csv','qa')]:
    for r in csv.DictReader(open('/kaggle/working/'+f)):
        ok = abs(float(r['f1']) - exp[(m, r['task'])]) < 0.002
        print('OK ' if ok else 'FAIL', m, r['task'],
              'f1', r['f1'][:6], 'auroc', r['auroc'][:6],
              'CI [', r['auroc_lo'][:5], ',', r['auroc_hi'][:5], ']')

!cd /kaggle/working && zip -qr ragtruth_auroc_ci.zip \
    ragtruth_full_preds ragtruth_qa_preds ragtruth_full_cis.csv ragtruth_qa_cis.csv

OK  full QA f1 0.7138 auroc 0.9144 CI [ 0.890 , 0.937 ]
OK  full Summary f1 0.5493 auroc 0.8235 CI [ 0.787 , 0.854 ]
OK  full Data2txt f1 0.8833 auroc 0.9103 CI [ 0.889 , 0.929 ]
OK  full whole f1 0.7911 auroc 0.9100 CI [ 0.898 , 0.921 ]
OK  qa QA f1 0.6796 auroc 0.8964 CI [ 0.863 , 0.925 ]
OK  qa Summary f1 0.4860 auroc 0.7663 CI [ 0.728 , 0.800 ]
OK  qa Data2txt f1 0.7715 auroc 0.5571 CI [ 0.519 , 0.593 ]
OK  qa whole f1 0.7070 auroc 0.8351 CI [ 0.819 , 0.850 ]
